In [9]:
import os
# os.environ['CUDA_VISIBLE_DEVICES'] = "1"

import sys
print(os.getcwd())
sys.path.append('/root/data1/Code/Mumu/FlagEmbedding/FlagEmbedding/visual/VISTA_Evaluation_FineTuning-main/evaluation_example_webqa/BGE-base')
import faiss
import torch
import logging
import datasets
import numpy as np
from tqdm import tqdm
from typing import Optional
from dataclasses import dataclass, field
from transformers import HfArgumentParser
# from FlagEmbedding import FlagModel
from flag_eva_token_new import Flag_bgev_model
# from flag_clip import Flag_clip
import json

/root/data1/Code/Mumu/FlagEmbedding/FlagEmbedding/visual/VISTA_Evaluation_FineTuning-main/evaluation_example_webqa/BGE-base


In [10]:
print(torch.cuda.is_available())

True


In [11]:
# parser = HfArgumentParser([Args])
# args: Args = parser.parse_args_into_dataclasses()[0]
resume_path = "/root/data1/Code/Multimodalqa/flod_save_models/fold_1_model/BGE_EVA_Token.pth"
# resume_path = "/root/data1/Code/Mumu/save_models_caption/checkpoint-2900/BGE_EVA_Token.pth"
image_dir = "/root/data1/Code/Multimodalqa/MMQA/final_dataset_images/"
# eval_data = datasets.load_dataset('json', data_files="/root/data5/MuMuQA/eval/test.json", split='train')
# # mm_it_corpus = datasets.load_dataset('json',  data_files="the_path_to/mm_it_corpus.jsonl", split='train')
# text_corpus = datasets.load_dataset('json', data_files="/root/data5/MuMuQA/eval/test.json", split='train')

model = Flag_bgev_model(model_name_bge = "BAAI/bge-base-en-v1.5",
                    model_name_eva = "EVA02-CLIP-B-16", # "EVA02-CLIP-B-16",
                    normlized = True,
                    eva_pretrained_path = "eva_clip",
                    resume_path=resume_path,
                    image_dir=image_dir,
                    )

True
cuda
----------using 2*GPUs----------


In [12]:
def search(model: Flag_bgev_model, queryQ,queryI, corpus,k:int = 100, batch_size: int = 1, max_length: int=512):
    # model.eval()
    with torch.no_grad():
        query_embedding = model.encode_queries([queryQ], batch_size=batch_size, max_length=max_length, query_type="text")
        text_corpus = corpus
        
        text_corpus_embeddings = model.encode_corpus_item(text_corpus, batch_size=batch_size, max_length=max_length, corpus_type='text')
        # print(text_corpus_embeddings.shape)
        # print(query_embedding.shape)

        similarity = query_embedding @ text_corpus_embeddings.T
        # print(query_embedding.shape)
        # print(all_embeddings.shape)
        # print(similarity.shape)
        # print(type(similarity))
        # similarity = similarity.cpu().numpy()
        similarity = similarity.squeeze()
        top_indices = np.argsort(-similarity)[:k]
        # 获取对应的句子
        top_sentences = [text_corpus[i] for i in top_indices]

        return top_sentences

In [13]:
def append_to_json_file(data, file_path):
    # 检查文件是否存在
    if os.path.exists(file_path):
        # 如果文件存在，读取当前的JSON数据
        with open(file_path, "r") as f:
            try:
                file_data = json.load(f)  # 读取已有的JSON文件内容
            except json.JSONDecodeError:
                file_data = []  # 如果文件是空的或者格式错误，初始化为空列表
    else:
        file_data = []  # 文件不存在时，初始化为空列表

    # 将新数据追加到文件数据中
    file_data.append(data)

    # 将更新后的数据重新写入JSON文件
    with open(file_path, "w") as f:
        json.dump(file_data, f, indent=4)

In [14]:
from my_utils import *
text_data = read_jsonl("/root/data1/Code/Multimodalqa/MMQA/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_texts.jsonl")
image_data = read_jsonl("/root/data1/Code/Multimodalqa/MMQA/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_images.jsonl")
# test_data =  read_json(args.test_path)
id2text = dict()
id2imagepath = dict()
for item in text_data:
    data_id =item["id"]
    id2text[data_id] = item["text"]
for item in image_data:
    data_id =item["id"]
    id2imagepath[data_id] = item["path"]

In [15]:
eval_data = datasets.load_dataset('json', data_files="/root/data1/Code/Multimodalqa/flod_save_models/fold_1_origineval.json", split='train')
    # mm_it_corpus = datasets.load_dataset('json',  data_files="the_path_to/mm_it_corpus.jsonl", split='train')
# text_corpus = datasets.load_dataset('json', data_files="/root/data1/Code/Mumu/processdata/retrive_goundtruth_ansnoNone.json", split='train')

In [16]:
print(len(eval_data))
# print(eval_data[0])
datas =[]
for idx,item in enumerate(tqdm(list(eval_data)[:])):
    pic_id = id2imagepath[item["supporting_context"][-1]["doc_id"]]
    ques = item["question"]
    txt = []
    for txt_item in item["metadata"]["text_doc_ids"]:
        txt.append(id2text[txt_item])
    candidate = []
    t_id = set()
    for t_item in item["supporting_context"]:
        if t_item["doc_part"] == "text":
            t_id.add(t_item["doc_id"])
    for t_item in t_id:
        candidate.append(id2text[t_item])
    top_sentences = search(
            model=model, 
            queryQ=ques, 
            queryI = pic_id,
            corpus = txt,
            k=100,
            batch_size=1, 
            max_length=512
        )
    data = {
            'data_id': item["qid"], 
            'question': ques, 
            'candidates': candidate,
            'retrieved': top_sentences,
        }
    append_to_json_file(data,'/root/data1/Code/Multimodalqa/flod_save_models/retrieve_data/modality/text_1_retrieval.json') 

131


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 131/131 [00:12<00:00, 10.14it/s]
